# 004 Summary And Embedding To ES

这是 RAG 知识库学习线的第四课。

上一课产物：

```text
chunks.json
```

本课目标：

```text
chunks.json
-> 可选摘要
-> Conan-embedding-v1 向量
-> Elasticsearch rag_chunks 索引
-> BM25 查询
-> 向量查询
```

学习目标：

1. 理解为什么 ES 同时保存原文、摘要、元数据和向量。
2. 使用 `Conan-embedding-v1` 生成 chunk embedding。
3. 根据 embedding 维度动态创建 ES `dense_vector` mapping。
4. 区分 BM25 查询和向量 kNN 查询。
5. 理解批量 embedding 和批量写入 ES 的工程边界。

注意：本课默认不自动批量写 ES。确认 ES 可连接后，再把 `RUN_ES_INDEXING` 改为 `True`。

## 1. 本课的位置

当前阶段：

```text
chunks.json
-> summary 可选
-> embedding
-> ES index
```

ES 在这里负责两件事：

```text
BM25 关键词检索
向量语义检索
```

Neo4j 还不会在本课出现。图谱会从第五课三元组抽取开始。

## 2. 导入依赖

本课使用：

```text
openai        -> 调 OpenAI-compatible embedding/chat 网关
elasticsearch -> 连接 ES
python-dotenv -> 读取 .env
```

In [4]:
import importlib.metadata
import json
import os
from hashlib import sha1
from pathlib import Path
from pprint import pprint
from time import sleep

from dotenv import load_dotenv
from elasticsearch import Elasticsearch, helpers
from openai import OpenAI

print('openai', importlib.metadata.version('openai'))
print('elasticsearch', importlib.metadata.version('elasticsearch'))

openai 2.36.0
elasticsearch 8.19.3


## 3. 加载配置和 chunks.json

如果第三课已经执行过，会读取：

```text
notebooks/rag/generated/{doc_id}/chunks.json
```

如果这里找不到文件，请先执行第三课。

In [5]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / 'requirements.txt').exists() and (path / 'notebooks').exists():
            return path
    return current

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env', override=False)

SAMPLE_PDF = PROJECT_ROOT / 'raw' / '北京市密云水库防御洪水方案.pdf'
if not SAMPLE_PDF.exists():
    raise FileNotFoundError(SAMPLE_PDF)

doc_id = sha1(SAMPLE_PDF.read_bytes()).hexdigest()[:16]
generated_dir = PROJECT_ROOT / 'notebooks' / 'rag' / 'generated' / doc_id
chunks_json_path = generated_dir / 'chunks.json'

if not chunks_json_path.exists():
    raise FileNotFoundError(f'请先执行第三课生成 chunks.json: {chunks_json_path}')

chunks = json.loads(chunks_json_path.read_text(encoding='utf-8'))

print('doc_id:', doc_id)
print('chunks_json_path:', chunks_json_path)
print('chunk_count:', len(chunks))
print('first_chunk_id:', chunks[0]['chunk_id'])
print('first_chunk_chars:', chunks[0]['char_count'])

doc_id: 63b7d4d0675426b5
chunks_json_path: /home/dev/bxc/fastapi-study/notebooks/rag/generated/63b7d4d0675426b5/chunks.json
chunk_count: 143
first_chunk_id: 63b7d4d0675426b5_chunk_0001
first_chunk_chars: 6


## 4. 配置模型网关和 ES

当前教学环境：

```text
model gateway: http://192.168.102.19:8082/v1
embedding model: Conan-embedding-v1
chat model: qwen2.5-0.5b-instruct
ES: http://10.20.20.45:9200
```

ES 当前可能存在网络不可达问题，所以后续 ES 写入单元会先检查连接状态。

In [6]:
RAG_CONFIG = {
    'model_base_url': os.getenv('RAG_MODEL_BASE_URL', 'http://192.168.102.19:8082/v1'),
    #'embedding_model': os.getenv('RAG_EMBEDDING_MODEL', 'qwen3-embedding'),
    'embedding_model': 'qwen3-embedding',
    'chat_model': os.getenv('RAG_CHAT_MODEL', 'qwen2.5-0.5b-instruct'),
    'es_addresses': os.getenv('RAG_ES_ADDRESSES', 'http://192.168.102.19:9200'),
    'es_user': os.getenv('RAG_ES_USER', 'elastic'),
    'es_password': os.getenv('RAG_ES_PASSWORD', 'elastic@2024'),
    'es_index': os.getenv('RAG_ES_INDEX', 'rag_chunks'),
}

safe_config = dict(RAG_CONFIG)
safe_config['es_password'] = '***'
pprint(safe_config)

model_client = OpenAI(api_key=os.getenv('RAG_MODEL_API_KEY', 'EMPTY'), base_url=RAG_CONFIG['model_base_url'])

{'chat_model': 'qwen2.5-0.5b-instruct',
 'embedding_model': 'qwen3-embedding',
 'es_addresses': 'http://192.168.102.19:9200',
 'es_index': 'rag_chunks',
 'es_password': '***',
 'es_user': 'elastic',
 'model_base_url': 'http://192.168.102.19:8082/v1'}


## 5. 单条 embedding 验证

先不要批量处理。先拿一个 chunk 调 embedding，观察向量维度。

这个维度后面要写入 ES mapping 的 `dims`。

In [7]:
def embed_texts(texts: list[str], model: str | None = None) -> list[list[float]]:
    model = model or RAG_CONFIG['embedding_model']
    response = model_client.embeddings.create(model=model, input=texts)
    return [item.embedding for item in response.data]

sample_text = chunks[0]['text']
sample_vector = embed_texts([sample_text])[0]
embedding_dims = len(sample_vector)

print('sample_chunk_id:', chunks[0]['chunk_id'])
print('sample_text_chars:', len(sample_text))
print('embedding_dims:', embedding_dims)
print('vector_preview:', sample_vector[:5])

sample_chunk_id: 63b7d4d0675426b5_chunk_0001
sample_text_chars: 6
embedding_dims: 1024
vector_preview: [-0.015763873234391212, -0.015365795232355595, -0.009593670256435871, -0.09171707928180695, 0.014012331143021584]


## 6. 可选：生成 chunk 摘要

摘要不是第一版检索必需项。

本课只演示对一个 chunk 生成摘要，避免一次性对所有 chunk 调模型。

后续如果要批量摘要，可以把这个函数用于循环，并把结果写回 `summary` 字段。

In [8]:
def summarize_chunk(text: str, max_chars: int = 700) -> str:
    prompt = (
        '请用一句中文概括下面文本，要求：\n'
        '1. 只输出摘要，不要输出 Markdown。\n'
        '2. 不要编造原文没有的信息。\n'
        '3. 控制在 80 字以内。\n\n'
        '文本：\n'
        + text[:max_chars]
    )
    response = model_client.chat.completions.create(
        model=RAG_CONFIG['chat_model'],
        temperature=0,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return response.choices[0].message.content.strip()

try:
    demo_summary = summarize_chunk(chunks[0]['text'])
    print('demo summary:', demo_summary)
except Exception as exc:
    demo_summary = ''
    print('summary failed:', type(exc).__name__, exc)

demo summary: 附件4：重要文件列表


## 7. 构造 ES 文档

ES 文档要同时保留：

```text
原文 text
可选 summary
vector
元数据
页码
chunk_id
```

注意：embedding 应该基于原文 `text`，不是只基于摘要。

In [9]:
def build_es_document(chunk: dict, vector: list[float], summary: str = '') -> dict:
    return {
        'doc_id': chunk['doc_id'],
        'chunk_id': chunk['chunk_id'],
        'chunk_no': chunk['chunk_no'],
        'file_name': chunk['file_name'],
        'page_start': chunk['page_start'],
        'page_end': chunk['page_end'],
        'text': chunk['text'],
        'summary': summary,
        'vector': vector,
        'char_count': chunk['char_count'],
        'metadata': chunk.get('metadata', {}),
    }

sample_es_doc = build_es_document(chunks[0], sample_vector, summary=demo_summary)

print('sample es doc keys:', sorted(sample_es_doc.keys()))
print('sample chunk_id:', sample_es_doc['chunk_id'])
print('sample vector dims:', len(sample_es_doc['vector']))
print('sample summary:', sample_es_doc['summary'])

sample es doc keys: ['char_count', 'chunk_id', 'chunk_no', 'doc_id', 'file_name', 'metadata', 'page_end', 'page_start', 'summary', 'text', 'vector']
sample chunk_id: 63b7d4d0675426b5_chunk_0001
sample vector dims: 1024
sample summary: 附件4：重要文件列表


## 8. 创建 ES client 并检查连通性

如果这一步失败，不要继续执行后面的建索引和写入单元。

常见原因：

```text
ES 地址不通
账号密码不对
当前机器网络无法访问 10.20.20.45:9200
```

In [10]:
def create_es_client() -> Elasticsearch:
    return Elasticsearch(
        hosts=RAG_CONFIG['es_addresses'],
        basic_auth=(RAG_CONFIG['es_user'], RAG_CONFIG['es_password']),
        request_timeout=10,
    )

es = create_es_client()
try:
    es_info = es.info()
    ES_READY = True
    print('ES connected:', es_info.get('version', {}).get('number'))
except Exception as exc:
    ES_READY = False
    print('ES connection failed:', type(exc).__name__, exc)

ES connected: 8.14.2


## 9. ES mapping 设计

`vector.dims` 必须等于 `Conan-embedding-v1` 返回的向量维度。

本环境实测维度为：

```text
1792
```

但 notebook 里不硬编码，而是用前面单条 embedding 得到的 `embedding_dims`。

In [11]:
def build_index_mapping(dims: int) -> dict:
    return {
        'mappings': {
            'properties': {
                'doc_id': {'type': 'keyword'},
                'chunk_id': {'type': 'keyword'},
                'chunk_no': {'type': 'integer'},
                'file_name': {'type': 'keyword'},
                'page_start': {'type': 'integer'},
                'page_end': {'type': 'integer'},
                'text': {'type': 'text', 'analyzer': 'standard'},
                'summary': {'type': 'text', 'analyzer': 'standard'},
                'char_count': {'type': 'integer'},
                'metadata': {'type': 'object', 'enabled': True},
                'vector': {
                    'type': 'dense_vector',
                    'dims': dims,
                    'index': True,
                    'similarity': 'cosine',
                },
            }
        }
    }

index_mapping = build_index_mapping(embedding_dims)
pprint(index_mapping)

{'mappings': {'properties': {'char_count': {'type': 'integer'},
                             'chunk_id': {'type': 'keyword'},
                             'chunk_no': {'type': 'integer'},
                             'doc_id': {'type': 'keyword'},
                             'file_name': {'type': 'keyword'},
                             'metadata': {'enabled': True, 'type': 'object'},
                             'page_end': {'type': 'integer'},
                             'page_start': {'type': 'integer'},
                             'summary': {'analyzer': 'standard',
                                         'type': 'text'},
                             'text': {'analyzer': 'standard', 'type': 'text'},
                             'vector': {'dims': 1024,
                                        'index': True,
                                        'similarity': 'cosine',
                                        'type': 'dense_vector'}}}}


## 10. 可选：创建索引

默认不会删除已有索引。

如果你要重建索引，可以手动设置：

```python
RESET_INDEX = True
```

In [12]:
RESET_INDEX = True

if not ES_READY:
    print('skip create index: ES is not connected')
else:
    index_name = RAG_CONFIG['es_index']
    if RESET_INDEX and es.indices.exists(index=index_name):
        es.indices.delete(index=index_name)
        print('deleted index:', index_name)

    if not es.indices.exists(index=index_name):
        es.indices.create(index=index_name, body=index_mapping)
        print('created index:', index_name)
    else:
        print('index exists:', index_name)

deleted index: rag_chunks
created index: rag_chunks


## 11. 可选：批量 embedding 并写入 ES

这一格会调用 embedding 模型并写 ES。

为了避免误操作，默认关闭：

```python
RUN_ES_INDEXING = False
```

确认 ES 可连后再改为 `True`。

In [13]:
RUN_ES_INDEXING = True
BATCH_SIZE = 4

def batched(items: list, batch_size: int):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]

def build_bulk_actions(chunks: list[dict], vectors: list[list[float]]) -> list[dict]:
    actions = []
    for chunk, vector in zip(chunks, vectors):
        doc = build_es_document(chunk, vector, summary=chunk.get('summary', ''))
        actions.append(
            {
                '_index': RAG_CONFIG['es_index'],
                '_id': chunk['chunk_id'],
                '_source': doc,
            }
        )
    return actions

if not RUN_ES_INDEXING:
    print('skip indexing: set RUN_ES_INDEXING = True to write ES')
elif not ES_READY:
    print('skip indexing: ES is not connected')
else:
    total_indexed = 0
    for chunk_batch in batched(chunks, BATCH_SIZE):
        texts = [chunk['text'] for chunk in chunk_batch]
        vectors = embed_texts(texts)
        actions = build_bulk_actions(chunk_batch, vectors)
        success, errors = helpers.bulk(es, actions, raise_on_error=False)
        total_indexed += success
        print('indexed batch:', success, 'errors:', len(errors))
        sleep(0.1)
    es.indices.refresh(index=RAG_CONFIG['es_index'])
    print('total_indexed:', total_indexed)

indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 4 errors: 0
indexed batch: 3 errors: 0
total_indexed: 143


## 12. BM25 查询示例

BM25 查询走 `match`，看的是 `text` 字段。

只有执行过上一格写入 ES 后，这一格才会有结果。

In [14]:
def bm25_search(query: str, size: int = 5):
    body = {
        'query': {
            'match': {
                'text': {
                    'query': query,
                }
            }
        },
        'highlight': {
            'fields': {
                'text': {
                    'pre_tags': ['<strong>'],
                    'post_tags': ['</strong>'],
                    'fragment_size': 160,
                }
            }
        },
    }
    return es.search(index=RAG_CONFIG['es_index'], body=body, size=size)

if not ES_READY:
    print('skip BM25 search: ES is not connected')
else:
    response = bm25_search('密云水库 防御 洪水')
    for hit in response['hits']['hits']:
        source = hit['_source']
        highlight = ''.join(hit.get('highlight', {}).get('text', []))
        print(hit['_score'], source['chunk_id'], source['page_start'])
        print(highlight or source['text'][:180])
        print('-' * 80)

7.575536 63b7d4d0675426b5_chunk_0037 32
科技推广中心保障雨、<strong>水</strong>、工情遥测系统正常运行；确

保报汛通讯通畅和报汛网络通畅。

后勤服务中心负责交通保障，确保<strong>水</strong>旱灾害<strong>防</strong><strong>御</strong>用车及

抢险物资运输。<strong>密</strong><strong>云</strong><strong>水</strong><strong>库</strong>管理处<strong>水</strong>旱灾害<strong>防</strong><strong>御</strong>工作领导小组加强与<strong>密</strong>

<strong>云</strong>区<strong>水</strong>务局对接联动，及时<strong>水</strong><strong>库</strong>下游属地政府通报<strong>水</strong>旱灾害

<strong>防</strong><strong>御</strong>工作信息，协助做好<strong>水</strong>旱灾害抢险救灾技术支撑工作，

及时报告<strong>防</strong><strong>洪</strong>调度方案和<strong>水</strong><strong>库</strong><strong>水</strong>雨情信息，泄<strong>洪</strong>期间加强信

息反馈及共享。<strong>水</strong>旱灾害<strong>防</strong><strong>御</strong>调度请示，见附表1；
<strong>水</strong>旱灾害<strong>防</strong><strong>御</strong>调度通知单，见附表2；
<strong>密</strong><strong>云</strong><strong>水</strong><strong>库</strong><strong>防</strong>汛调度令，见附表3;
<strong>密</strong><strong>云</strong><strong>水</strong><strong>库</strong>调度概化图，见附图6；
<str

/tmp/ipykernel_3776728/2386880366.py:20: DeprecationWarning: Received 'size' via a specific parameter in the presence of a 'body' parameter, which is deprecated and will be removed in a future version. Instead, use only 'body' or only specific parameters.
  return es.search(index=RAG_CONFIG['es_index'], body=body, size=size)


## 13. 向量查询示例

向量查询流程：

```text
用户问题
-> Conan-embedding-v1 生成 query vector
-> ES kNN 查询 vector 字段
```

它适合语义相近但关键词不完全一致的问题。

In [15]:
def vector_search(query: str, k: int = 5, num_candidates: int = 50):
    query_vector = embed_texts([query])[0]
    body = {
        'knn': {
            'field': 'vector',
            'query_vector': query_vector,
            'k': k,
            'num_candidates': num_candidates,
        },
        'size': k,
    }
    return es.search(index=RAG_CONFIG['es_index'], body=body)

if not ES_READY:
    print('skip vector search: ES is not connected')
else:
    response = vector_search('密云水库发生洪水时如何调度')
    for hit in response['hits']['hits']:
        source = hit['_source']
        print(hit['_score'], source['chunk_id'], source['page_start'])
        print(source['text'][:180])
        print('-' * 80)

0.8908596 63b7d4d0675426b5_chunk_0002 3
目 录

2024 年北京市密云水库洪水调度方案................................................................ 1

2024 年北京市密云水库防洪抢险预案................................................................ 63
--------------------------------------------------------------------------------
0.88492036 63b7d4d0675426b5_chunk_0030 25
6 月1 日至9 月30 日，七孔桥节制闸调度须服从密云水

库水旱灾害防御调度。

（2）调节池调度规程
6 月1 日至9 月30 日，调节池挡水闸闸门全开，调节池
最高水位不超过90.50m，以确保小西库上游农民耕地不被洪

水淹没，小西库上游洪水入调节池后优先引入京密引水渠，

若京密引水渠引水流量不能满足泄洪要求，报北京市水务局

批准后开启调节池泄
--------------------------------------------------------------------------------
0.8802161 63b7d4d0675426b5_chunk_0020 15
短期：每三小时进行滚动预报，实现预报成果动态更新。

中期：每日进行滚动预报，实现预报成果动态更新。

长期：每三日进行滚动预报，实现预报成果动态更新。

4.2.3 预报成果

洪水预报成果包括：场次降雨洪峰流量及峰现时间、入

库洪量、水库最高水位及出现时间。最终预报成果以北京市

水文总站预报结果为主。

4.3 洪水调度安排

相机实施预报调度，并根
--------------------------------------------------------------------------------
0.87785816 63b7d4d0675426b5_chunk_0018 13
1550m3/s。
（3）密云水库为下游错峰时间不超过26h。
（4）错峰过后按正常调度方案运行。
3

## 14. 本课小结

本课完成了：

```text
chunks.json
-> sample embedding
-> ES mapping
-> 可选批量写 ES
-> BM25 查询
-> 向量查询
```

关键结论：

```text
ES 里不要只存 vector。
必须同时存 text、metadata、page_start/page_end、chunk_id。
```

原因是：

```text
vector 用来召回
text 用来展示和回答
metadata 用来过滤和追溯
chunk_id 用来和 Neo4j 证据对齐
```

下一课会进入：

```text
chunks.json
-> qwen2.5-0.5b-instruct
-> 三元组抽取
-> triples.json
```

## 15. 练习

请你回答：

1. 为什么 ES 文档里必须保存 `chunk_id`？
2. 为什么 embedding 应该基于原文 `text`，而不是只基于摘要？
3. BM25 和向量检索分别更适合什么问题？
4. 如果 ES 连接失败，应该先排查地址、账号密码，还是 notebook 代码？